# Vaginal community state type (CST) estimation

This notebook follows the structure of `refine_taxonomy_cst_heatmap.ipynb`, adapted to the PacBio HiFi workflow outputs. It loads the ASV table, species taxonomy, and sample manifest; collapses ASVs by taxon; estimates CSTs from dominant *Lactobacillus* species; joins clinical metadata; and exports a clustered heatmap and analysis tables.

CST calls are sequence-based estimates and should not be interpreted as clinical diagnoses.

In [ ]:
from pathlib import Path
import os
os.environ.setdefault('MPLCONFIGDIR', '/tmp/pacbio_cst_matplotlib')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import pdist, squareform

WORKFLOW_RESULTS = Path('/home/rare/arlen/Pacbio_colombian_vaginal_microbiome/results_HiFi-16S-workflow')
INPUT_DIR = WORKFLOW_RESULTS / '02_microbiome_analyst_input_files'
ASV_TABLE = INPUT_DIR / 'asv_table_microbiomeanalyst.csv'
TAXONOMY_TABLE = INPUT_DIR / 'taxonomy_microbiomeanalyst.csv'
MANIFEST = INPUT_DIR / 'manifest.csv'
OUTPUT_DIR = WORKFLOW_RESULTS / '03_cst_estimation'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DOMINANT_CST_THRESHOLD = 0.30
TOP_N_TAXA = 40
pd.set_option('display.max_columns', 120)

## Load and validate inputs

In [ ]:
for path in (ASV_TABLE, TAXONOMY_TABLE, MANIFEST):
    if not path.is_file():
        raise FileNotFoundError(path)

asv = pd.read_csv(ASV_TABLE, index_col=0)
taxonomy = pd.read_csv(TAXONOMY_TABLE, sep='\t', index_col=0)
metadata = pd.read_csv(MANIFEST, sep='\t', dtype=str).set_index('#NAME')

asv.index = asv.index.astype(str)
taxonomy.index = taxonomy.index.astype(str)
asv = asv.apply(pd.to_numeric, errors='coerce').fillna(0)

missing_taxonomy = asv.index.difference(taxonomy.index)
missing_metadata = asv.columns.difference(metadata.index)
if len(missing_taxonomy):
    raise ValueError(f'{len(missing_taxonomy)} ASVs lack taxonomy')
if len(missing_metadata):
    raise ValueError(f'{len(missing_metadata)} samples lack metadata: {missing_metadata.tolist()}')

print(f'ASV table: {asv.shape[0]:,} ASVs x {asv.shape[1]:,} samples')
print(f'Taxonomy rows: {len(taxonomy):,}')
print(f'Metadata rows: {len(metadata):,}')

## Build display taxonomy and collapse ASVs

The deepest non-empty rank is used for each ASV. Species names are preferred; lower-resolution labels retain their rank to avoid implying species-level resolution.

In [ ]:
taxonomy.columns = taxonomy.columns.str.strip()
rank_order = ['Species', 'Genus', 'Family', 'Order', 'Class', 'Phylum', 'Kingdom']

def clean_taxon(value):
    if pd.isna(value):
        return None
    value = str(value).strip()
    return None if value.lower() in {'', 'nan', 'none', 'unclassified', 'uncultured'} else value

def display_taxon(row):
    for rank in rank_order:
        value = clean_taxon(row.get(rank))
        if value:
            return value if rank == 'Species' else f'{value} ({rank.lower()})'
    return 'Unclassified'

taxonomy['display_taxon'] = taxonomy.apply(display_taxon, axis=1)
taxonomy[['display_taxon'] + [c for c in rank_order if c in taxonomy]].to_csv(
    OUTPUT_DIR / 'taxonomy_with_display_names.tsv', sep='\t'
)
taxon_counts = asv.assign(display_taxon=taxonomy.loc[asv.index, 'display_taxon']).groupby('display_taxon').sum()
sample_totals = taxon_counts.sum(axis=0)
if (sample_totals == 0).any():
    raise ValueError(f'Zero-count samples: {sample_totals.index[sample_totals == 0].tolist()}')
relative_abundance = taxon_counts.div(sample_totals, axis=1)
relative_abundance.to_csv(OUTPUT_DIR / 'taxon_relative_abundance.tsv', sep='\t')
(relative_abundance * 100).to_csv(OUTPUT_DIR / 'taxon_relative_abundance_percent.tsv', sep='\t')
print(f'Collapsed taxa: {len(relative_abundance):,}')
(relative_abundance.mean(axis=1).sort_values(ascending=False).head(15) * 100).round(2).to_frame('mean_percent')

## Estimate CSTs

CST I, II, III, and V are called when their marker species reaches 30% relative abundance and is at least 80% of the most abundant taxon. Other profiles are assigned CST IV, with an exploratory IV-A/IV-B detail based on the dominant taxon.

In [ ]:
CST_MARKERS = {
    'Lactobacillus crispatus': 'CST I',
    'Lactobacillus gasseri': 'CST II',
    'Lactobacillus iners': 'CST III',
    'Lactobacillus jensenii': 'CST V',
}
BV_KEYWORDS = ('Gardnerella', 'Fannyhessea', 'Atopobium', 'Megasphaera', 'Sneathia', 'Prevotella', 'Mobiluncus', 'Dialister', 'Anaerococcus', 'Peptoniphilus')

def assign_cst(sample_ra):
    top_taxon = sample_ra.idxmax()
    top_abundance = float(sample_ra.max())
    marker_values = {taxon: float(sample_ra.get(taxon, 0)) for taxon in CST_MARKERS}
    best_marker = max(marker_values, key=marker_values.get)
    best_value = marker_values[best_marker]
    lactobacillus_total = float(sample_ra[sample_ra.index.str.startswith('Lactobacillus ')].sum())
    if best_value >= DOMINANT_CST_THRESHOLD and best_value >= 0.80 * top_abundance:
        cst = CST_MARKERS[best_marker]
        detail = f'{cst}: {best_marker} dominated'
    else:
        cst = 'CST IV'
        subtype = 'IV-B: anaerobe/BV-associated' if any(x in top_taxon for x in BV_KEYWORDS) else 'IV-A: diverse'
        detail = f'CST {subtype}; top taxon {top_taxon}'
    return pd.Series({
        'CST': cst, 'CST_detail': detail, 'dominant_taxon': top_taxon,
        'dominant_relative_abundance': top_abundance, 'cst_marker_taxon': best_marker,
        'cst_marker_relative_abundance': best_value,
        'lactobacillus_relative_abundance': lactobacillus_total,
    })

assignments = relative_abundance.apply(assign_cst, axis=0).T
assignments.index.name = 'sample-id'
annotation_columns = [c for c in ['condition', 'Disbiosis'] if c in metadata.columns]
assignments = assignments.join(metadata[annotation_columns], how='left')
assignments.to_csv(OUTPUT_DIR / 'sample_cst_assignments.tsv', sep='\t')
print(assignments['CST'].value_counts().to_string())
assignments

## Cluster samples and plot the CST heatmap

In [ ]:
species_ids = taxonomy.index[taxonomy['Species'].map(clean_taxon).notna()]
species_taxa = taxonomy.loc[species_ids, 'display_taxon'].unique()
species_ra = relative_abundance.loc[relative_abundance.index.intersection(species_taxa)]
mandatory = [x for x in CST_MARKERS if x in species_ra.index]
top = species_ra.mean(axis=1).sort_values(ascending=False).head(TOP_N_TAXA).index.tolist()
plot_taxa = list(dict.fromkeys(mandatory + top))[:TOP_N_TAXA]
plot_pct = species_ra.loc[plot_taxa] * 100
plot_pct = plot_pct.loc[plot_pct.max(axis=1).gt(0)]

if plot_pct.shape[1] > 1:
    distance = squareform(pdist(plot_pct.T, metric='braycurtis'))
    csts = assignments.reindex(plot_pct.columns)['CST'].fillna('Unassigned').to_numpy()
    distance += (csts[:, None] != csts[None, :]).astype(float) * 2.0
    col_linkage = linkage(squareform(distance, checks=False), method='average')
else:
    col_linkage = None

CST_PALETTE = {'CST I': '#fb7185', 'CST II': '#a3ad23', 'CST III': '#41b6b3', 'CST IV': '#a78bfa', 'CST V': '#f59e0b'}
CONDITION_PALETTE = dict(zip(sorted(metadata.get('condition', pd.Series(dtype=str)).dropna().unique()), sns.color_palette('Set2').as_hex()))
col_colors = pd.DataFrame(index=plot_pct.columns)
col_colors['CST'] = assignments.reindex(plot_pct.columns)['CST'].map(CST_PALETTE).fillna('#d9d9d9')
if 'condition' in assignments:
    col_colors['Condition'] = assignments.reindex(plot_pct.columns)['condition'].map(CONDITION_PALETTE).fillna('#d9d9d9')

sns.set_theme(context='paper', style='white', font_scale=0.9)
g = sns.clustermap(
    plot_pct, row_cluster=False, col_cluster=col_linkage is not None, col_linkage=col_linkage,
    col_colors=col_colors, cmap='viridis', vmin=0, vmax=100,
    figsize=(max(13, 0.23 * plot_pct.shape[1] + 5), max(10, 0.24 * plot_pct.shape[0] + 3.5)),
    dendrogram_ratio=(0.015, 0.22), colors_ratio=(0.015, 0.035),
    cbar_pos=(0.035, 0.70, 0.022, 0.18),
    cbar_kws={'label': 'Relative abundance (%)', 'ticks': [0, 20, 40, 60, 80, 100]},
)
g.ax_heatmap.set_xlabel('')
g.ax_heatmap.set_ylabel('Species')
g.ax_heatmap.yaxis.tick_right()
g.ax_heatmap.tick_params(axis='x', labelsize=7, rotation=90)
g.ax_heatmap.tick_params(axis='y', labelsize=8)
observed_csts = [cst for cst in CST_PALETTE if cst in assignments.reindex(plot_pct.columns)['CST'].dropna().unique()]
handles = [Patch(facecolor=CST_PALETTE[cst], edgecolor='none', label=cst) for cst in observed_csts]
g.fig.legend(handles=handles, title='CST', loc='upper right', bbox_to_anchor=(0.985, 0.84), frameon=False)
if 'condition' in assignments:
    observed_conditions = [condition for condition in CONDITION_PALETTE if condition in assignments.reindex(plot_pct.columns)['condition'].dropna().unique()]
    condition_handles = [Patch(facecolor=CONDITION_PALETTE[condition], edgecolor='none', label=condition) for condition in observed_conditions]
    g.fig.legend(handles=condition_handles, title='Condition', loc='upper right', bbox_to_anchor=(0.985, 0.68), frameon=False)
g.fig.suptitle(f'PacBio HiFi species-level CST heatmap (top {TOP_N_TAXA} species)', y=0.995)
g.fig.subplots_adjust(top=0.94, bottom=0.10, left=0.16, right=0.82)
g.cax.set_position([0.035, 0.70, 0.022, 0.18])

sample_order = plot_pct.columns[g.dendrogram_col.reordered_ind].tolist() if col_linkage is not None else plot_pct.columns.tolist()
pd.Series(sample_order, name='sample-id').to_csv(OUTPUT_DIR / 'clustered_heatmap_sample_order.tsv', sep='\t', index=False)
plot_pct.loc[:, sample_order].to_csv(OUTPUT_DIR / 'species_heatmap_relative_abundance_percent.tsv', sep='\t')
g.fig.savefig(OUTPUT_DIR / 'cst_heatmap.png', dpi=600, bbox_inches='tight')
g.fig.savefig(OUTPUT_DIR / 'cst_heatmap.pdf', bbox_inches='tight')
plt.show()

## Outputs

All products are written to `03_cst_estimation/`: taxonomy audit, relative-abundance matrices, sample CST assignments, plotted sample order, heatmap matrix, PNG, and PDF.

In [ ]:
for path in sorted(OUTPUT_DIR.glob('*')):
    print(path)